# Filter out categorised professions by count

1. I combined a list of all professions from debiaswe and 100-years-of-stereotypes.
2. I manually curated them and counted their (relevant) occurences in the overall dataset
    - this involved sometimes ensuring noun-usage and other times whole-wordedness
3. Then I partially manually categorised them into the 43 ILO categories
    - some decisions remain to be made about what to include in the 44th "Unemployed" category

Now I want to keep all the top 5 occupations in each category. 

In [9]:
import json
from pathlib import Path

root_dir = Path.cwd().parent.parent
cat_file = root_dir / "data" / "occupations" / "profession_category_to_professions.json"
final_profs_file = root_dir / "data" / "occupations" / "filtered_professions.json"
count_file = root_dir / "data" / "dolci" / "dolci_profession_counts.json"

with open(cat_file, "r") as f:
    data = json.load(f)
with open(count_file, "r") as f:
    counts = json.load(f)

In [2]:
import pandas as pd

# Create a list to store the data
data_list = []

# Iterate through each category and its professions
for category, professions in data.items():
    # Handle cases where professions might be a string instead of a list
    if isinstance(professions, str):
        professions = [professions]
    
    # For each profession in the category
    for profession in professions:
        # Find the count for this profession
        count = next((item['count'] for item in counts if item['profession'] == profession), 0)
        data_list.append({
            'profession': profession,
            'category': category,
            'count': count
        })

# Create the dataframe
df = pd.DataFrame(data_list)
df.head()

,profession,category,count
0,actor,"legal, social and cultural professionals",139454
1,actress,"legal, social and cultural professionals",2589
2,advocate,"legal, social and cultural professionals",3099
3,anthropologist,"legal, social and cultural professionals",1133
4,archbishop,"legal, social and cultural professionals",140


In [6]:
df["category"].value_counts()

category
legal, social and cultural professionals                                            80
legal, social, cultural and related associate professionals                         19
science and engineering professionals                                               17
chief executives, senior officials and legislators                                  16
health professionals                                                                15
teaching professionals                                                              15
protective services workers                                                         13
business and administration professionals                                           12
personal service workers                                                            10
administrative and commercial managers                                               9
business and administration associate professionals                                  7
commissioned armed forces officers

In [3]:
# keep top-5 professions by count within each category (count > 500)
df_filtered = df[df["count"] > 500]
df_top5 = (
    df_filtered.sort_values(['category', 'count'], ascending=[True, False])
      .groupby('category', group_keys=False)
      .head(5)
      .reset_index(drop=True)
)

len(df_top5)

92

In [4]:
# save top5 df to json
output_file = root_dir / "data" / "occupations" / "top5_professions_by_category.json"
df_top5.to_json(output_file, orient='records', indent=4)

In [7]:
df_top5.head()

,profession,category,count
0,manager,administrative and commercial managers,22965
1,director,administrative and commercial managers,21217
2,entrepreneur,administrative and commercial managers,8741
3,administrator,administrative and commercial managers,4149
4,gardener,"agricultural, forestry and fishery labourers",801


In [10]:
# Save df_top5 professions to a list and write to json
final_professions = df_top5['profession'].unique().tolist()
with open(final_profs_file, 'w') as f:
    json.dump(final_professions, f, indent=2)
